## Logistics Simulation & What-If Scenarios

In [17]:
import numpy as np
import pandas as pd
import yaml
from pathlib import Path

# reload config (you edited config.yaml)
with open("../config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

# load processed data created in Steps 3 & 4
lanes   = pd.read_csv("../data/processed/lanes_generated.csv")
demand  = pd.read_csv("../data/processed/demand_simulated.csv")

# quick sanity checks
print("lanes:", lanes.shape)
print("demand:", demand.shape)
assert {"retailer_id","sku_id","week","demand_units"}.issubset(demand.columns)
assert {"from_type","to_type","base_lead_days","transport_cost_per_unit"}.issubset(lanes.columns)

lanes: (6, 7)
demand: (936, 4)


In [18]:
def simulate_shipment(retailer_id, sku_id, week, units, lanes_df, reliability_cfg):
    """
    Simulate one shipment flow from supplier -> factory -> warehouse -> retailer
    Returns total_lead_days, delivered_on_time (bool), total_cost
    """
    total_lead = 0
    total_cost = 0
    route = ["supplier", "factory", "warehouse", "retail"]

    # Each leg contributes base lead and cost
    for i in range(len(route) - 1):
        leg_from, leg_to = route[i], route[i + 1]
        lane = lanes_df[(lanes_df.from_type == leg_from) & (lanes_df.to_type == leg_to)].sample(1).iloc[0]
        base_lt = lane["base_lead_days"]
        cost = lane["transport_cost_per_unit"] * units

        # Apply reliability: sometimes delay occurs
        if np.random.rand() > reliability_cfg["lane_reliability"]:
            delay = np.random.randint(1, reliability_cfg["max_delay_days"] + 1)
        else:
            delay = 0

        total_lead += base_lt + delay
        total_cost += cost

    # On-time if total_lead <= expected (mean lead_time * 1.1)
    expected_lead =  (len(route) - 1) * reliability_cfg["expected_base_days"]
    on_time = total_lead <= expected_lead * 1.1
    return total_lead, on_time, total_cost

In [19]:
rows = []

for _, r in demand.iterrows():
    lead, ontime, cost = simulate_shipment(
        r["retailer_id"],
        r["sku_id"],
        int(r["week"]),
        float(r["demand_units"]),
        lanes,
        reliability_cfg
    )
    rows.append({
        "retailer_id": r["retailer_id"],
        "sku_id": r["sku_id"],
        "week": int(r["week"]),
        "demand_units": float(r["demand_units"]),
        "realized_lead_days": int(lead),
        "delivered_on_time": bool(ontime),
        "transport_cost": float(cost),
    })

ship_df = pd.DataFrame(rows)
ship_df.head()

,retailer_id,sku_id,week,demand_units,realized_lead_days,delivered_on_time,transport_cost
0,RET_1,SKU_1,1,0.0,16,True,0.0
1,RET_1,SKU_1,2,0.0,16,True,0.0
2,RET_1,SKU_1,3,0.0,16,True,0.0
3,RET_1,SKU_1,4,0.0,19,False,0.0
4,RET_1,SKU_1,5,0.0,16,True,0.0


In [20]:
summary = {
    "avg_lead_days": round(ship_df["realized_lead_days"].mean(), 2),
    "on_time_rate": round(ship_df["delivered_on_time"].mean() * 100, 2),
    "total_cost": round(ship_df["transport_cost"].sum(), 2)
}
print(summary)

{'avg_lead_days': np.float64(16.46), 'on_time_rate': np.float64(79.7), 'total_cost': np.float64(27580.41)}


In [21]:
out = Path("../data/processed/shipments_simulated.csv")
out.parent.mkdir(parents=True, exist_ok=True)
ship_df.to_csv(out, index=False)
print(f"✅ Shipment simulation saved to {out}")

✅ Shipment simulation saved to ..\data\processed\shipments_simulated.csv
